# Initialisatie van tien geleerde schaalkeuzes — start ×1 tegenover start ×64

**SVHN · CNN-8 · E4M3 → E2M3 → E2M1 · A32 · FP32-masters · B32**

De nieuwe pilot start alle 4.080 schaalparameters op **θ = 0 (factor ×1)**. De bestaande referentie begon op **θ = 6 (factor ×64)**. De tien mogelijkheden ×1–×512 blijven in beide gevallen vrij leerbaar. De bedoelde trainingswijziging is uitsluitend de beginwaarde.

×1 betekent **S_E2 = S_E4**, niet een absolute schaal van 1. De E4M3-bronschaal wordt uit de actuele gewichten berekend. E2M3 en E2M1 delen dezelfde lagere schaal en behouden hun bestaande exponent- en mantissatruncatie.

**Deze pilot stopt uiterlijk na vijf epochs.** Zij laat zien hoe de parameters aanvankelijk bewegen. Zij bepaalt niet of een verre start uiteindelijk naar dezelfde oplossing convergeert. De figuren gebruiken opgeslagen JSON/CSV; er wordt geen training gestart en geen checkpoint geladen.

In [ ]:
from pathlib import Path
RUN_DIR = Path('.')  # De runner vult in de runkopie het exacte resultaatpad in.
SAVE_FIGURES = False  # True schrijft naar een nieuwe analysemap; ruwe resultaten blijven behouden.


In [ ]:
import sys
import importlib.util
import pandas as pd
import matplotlib.pyplot as plt
try:
    from IPython.display import Markdown, display
except ImportError:
    class Markdown(str): pass
    def display(value): print(value)
helper_path = RUN_DIR.expanduser().resolve() / 'notebook_support.py'
if not helper_path.is_file():
    raise FileNotFoundError('Zet RUN_DIR op de resultatenmap van de init×1-pilot: ' + str(helper_path.parent))
spec = importlib.util.spec_from_file_location('_mx_scale10_init1_pilot_analysis', helper_path)
ns = importlib.util.module_from_spec(spec)
sys.modules[spec.name] = ns
spec.loader.exec_module(ns)
run = ns.load_run(RUN_DIR)
figures = {}
display(Markdown(ns.status_markdown(run)))
display(ns.diagnostics_table(run))
display(ns.protocol_table(run))


## 1. Accuracy gedurende de vijf pilotepochs

Alle drie precisiepanelen delen hun y-as. Het bereik volgt de werkelijk getoonde waarden: als de nieuwe initialisatie lage accuracies oplevert, blijven die zichtbaar. Alleen het beschikbare gemeenschappelijke epochvenster wordt vergeleken. Er wordt geen epoch-100-score getoond of voorspeld.

In [ ]:
figures['accuracy_verloop'] = ns.plot_initialization_accuracy(run)
display(figures['accuracy_verloop'])
plt.close(figures['accuracy_verloop'])


## 2. Primair piloteindpunt: epoch 5

De vergelijking gebruikt voor beide initialisaties epoch 5. Zij blijft leeg zolang die epoch niet correct beschikbaar is. De gekoppelde punten hebben een duidelijk gemarkeerde ingezoomde as; de twee checkpointfiguren gebruiken dezelfde grenzen.

In [ ]:
display(ns.initialization_checkpoint_table(run, 'endpoint').drop(columns=['weight_bits']).round({'Testaccuracy [%]': 6}))
figures['accuracy_epoch5'] = ns.plot_initialization_checkpoints(run, 'endpoint')
display(figures['accuracy_epoch5'])
plt.close(figures['accuracy_epoch5'])


## 3. Aanvullend: het E4M3-beste checkpoint

Per variant wordt de eerste hoogste E4M3-testaccuracy in hetzelfde beschikbare venster 1–5 gekozen. Alle drie precisies komen uit die ene geselecteerde epoch. De twee varianten mogen een verschillende beste epoch hebben. Selectie op de officiële testset is geen onafhankelijke validatieschatting.

In [ ]:
display(ns.initialization_checkpoint_table(run, 'best').drop(columns=['weight_bits']).round({'Testaccuracy [%]': 6}))
figures['accuracy_best_E4M3'] = ns.plot_initialization_checkpoints(run, 'best')
display(figures['accuracy_best_E4M3'])
plt.close(figures['accuracy_best_E4M3'])


## 4. Evolutie van alle tien schaalfactoren

Epoch 0 is de werkelijk opgeslagen initialisatie. De heatmap houdt alle factoren zichtbaar, inclusief nulpercentages. Een factor kan gelijk blijven terwijl θ al beweegt binnen het bijbehorende afrondingsinterval.

In [ ]:
figures['factoren_per_epoch'] = ns.plot_factor_evolution(run)
display(figures['factoren_per_epoch'])
plt.close(figures['factoren_per_epoch'])


## 5. Groepen en percentages aan het begin en einde

De staafgrafieken beginnen op nul en tonen aantallen én aandelen. De historische eindverdeling gebruikt dezelfde afgeronde epoch als de nieuwe pilot. De E4M3-bronschaal blijft hierbij dynamisch.

In [ ]:
display(ns.factor_distribution_table(run, 0).round({'Aandeel [%]': 4}))
display(ns.factor_distribution_table(run).round({'Aandeel [%]': 4}))
figures['factoren_aantallen'] = ns.plot_factor_counts(run)
display(figures['factoren_aantallen'])
plt.close(figures['factoren_aantallen'])


## 6. Continue θ-beweging vóór een harde factorwissel

De optimizer past een continue FP32-parameter aan; de voorwaartse stap rondt die naar een integeroffset af. De schaal-LR blijft 0,001. Starten op θ=0 ligt ver van θ=6; vijf epochs zijn daarom geen voldoende grond om te besluiten dat de uiteindelijke oplossing niet bereikbaar is. Er geldt geen veronderstelde harde Adam-stapgrens.

In [ ]:
theta = ns.theta_summary_table(run)
display(theta.loc[theta.Laag.eq('Alle lagen')].round(6))
figures['theta_totaal'] = ns.plot_theta_evolution(run)
display(figures['theta_totaal'])
plt.close(figures['theta_totaal'])


## 7. Continue parameters per MX-laag

De zes panelen delen hun y-as. Gemiddelde, P05–P95 en minimum/maximum tonen of de beweging in bepaalde lagen zit. Er worden geen conclusies over vastlopen getrokken uit uitsluitend de harde schaalkeuzes.

In [ ]:
figures['theta_per_laag'] = ns.plot_theta_by_layer(run)
display(figures['theta_per_laag'])
plt.close(figures['theta_per_laag'])

# Per laag en epoch: een harde ×1-keuze is iets anders dan exact θ=0.
display(theta.loc[theta.Laag.ne('Alle lagen'), ['Epoch', 'Laag', 'Gemiddelde θ', 'Min θ', 'Max θ', 'Exact θ=0 [%]', 'Factor ×1 [%]']].round(6))


## 8. ×1 kiezen is niet hetzelfde als op de ondergrens vastzitten

Factor ×1 geldt zolang θ naar offset 0 wordt afgerond. **Exact θ=0** betekent dat de parameter op de projectiegrens staat. Adam kan θ vanuit 0 omhoog bewegen; updates onder 0 worden terug op de ondergrens geprojecteerd. Eén groep kan binnen een epoch meermaals worden geprojecteerd.

In [ ]:
figures['grenzen_en_projecties'] = ns.plot_initialization_boundaries(run)
display(figures['grenzen_en_projecties'])
plt.close(figures['grenzen_en_projecties'])


## 9. Gradients, echte updates en omschakelingen

Vergelijk het aantal gewijzigde FP32-parameters, de netto θ-verplaatsing, harde omschakelingen en de geregistreerde gradientnorm. De gradientfiguur toont de grootste **laag**-norm over alle batches; zij is geen globale gradientnorm van het hele netwerk. Flips en unieke wisselende groepen worden apart geteld.

In [ ]:
figures['updates_en_gradients'] = ns.plot_initialization_updates(run)
display(figures['updates_en_gradients'])
plt.close(figures['updates_en_gradients'])


## 10. Kosten, duur en interpretatie

De gewijzigde beginwaarde voegt geen extra schaalopties, parameters of passes toe. De opslag hieronder is analytisch; het zijn geen gemeten hardware-energiecijfers. Vergelijk de gemeten duur descriptief met de bestaande pilot.

**Beoordeling:** bekijk of θ vanaf ×1 beweegt, of de verdeling verschuift, en hoe de drie accuracies volgen. Een weinig bewegende verdeling na vijf epochs kan samenhangen met de afstand tot andere offsets, de leersnelheid of projectie op de grens. Een slechte korte run bewijst niet dat ×64 globaal optimaal is. Gelijke seeds en gecontroleerde datavolgordes geven bij de bestaande CUDA-instellingen geen bitexacte reproduceerbaarheid.

In [ ]:
display(ns.initialization_cost_table(run))
if not run.times.empty:
    display(run.times[['epoch', 'train_seconds', 'test_seconds', 'total_epoch_seconds']].round(3))
display(Markdown(ns.scale_learning_markdown(run)))


## Optioneel: figuren bewaren

Deze notebook blijft standaard alleen lezen en weergeven. Zet bovenaan `SAVE_FIGURES = True` om de figuren en bronverwijzingen in een nieuwe analysemap te bewaren.

In [ ]:
if SAVE_FIGURES:
    destination = ns.save_figures(figures, run)
    print('Figuren opgeslagen in:', destination)
else:
    print('Alleen gelezen en weergegeven; bestaande resultaten zijn ongewijzigd.')
